# Fraud Detection Project - Phase 1 Data Audit

This notebook is the first technical checkpoint in the fraud detection project. Its purpose is to perform an initial audit of the dataset before any cleaning, feature engineering, anomaly detection, or predictive modeling begins.

At this stage, the goal is not to produce insights too early or build models prematurely. Instead, the objective is to establish control over the dataset by answering several foundational questions:

- Did the dataset load correctly?
- What columns are available?
- What data types are currently assigned?
- Are there any missing values?
- What does each variable appear to represent?
- How is the fraud label distributed?
- What does the time variable (`step`) actually look like?

This notebook creates the structural understanding required for all later phases of the project. A strong audit at the beginning reduces downstream errors and improves the credibility of later analysis.

## Importing libraries and loading utilities

This section imports the Python libraries and helper functions that will be used throughout the notebook.

The purpose of this step is to establish the working environment for the analysis. We import:
- `Path` for file path handling,
- `pandas` for tabular data inspection,
- `numpy` for numerical operations,
- `load_sample` from the project source code so we can safely load only a subset of the dataset.

Using a sample instead of the full file is an intentional technical choice. Since the dataset is very large, loading a smaller portion first allows us to validate schema, inspect structure, and test code without unnecessary memory pressure.

We also configure pandas display options so the notebook output is easier to read. This improves interpretability when viewing wide tables or decimal values.

In [16]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Ensure project root is on sys.path when running from notebooks/
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_PATH = PROJECT_ROOT / "data/raw/paysim dataset.csv"

from src.load_data import load_sample

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Loading a sample of the dataset

In this step, we load a small sample of the transaction dataset rather than the entire file.

This matters for two reasons:
1. It allows us to verify that the file can be read successfully.
2. It gives us a fast and memory-safe way to inspect the dataset before deciding how to process the full dataset later.

At this stage, we are not yet trying to analyze the full data distribution or train any model. The goal is simply to make first contact with the data and confirm that the structure is consistent with expectations.

The expected result is a DataFrame containing a manageable number of rows that we can inspect interactively.

In [17]:
df = load_sample(nrows=10000, path=RAW_PATH)
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,"9,839.64",C1231006815,"170,136.00","160,296.36",M1979787155,0.00,0.00,0,0
1,1,PAYMENT,"1,864.28",C1666544295,"21,249.00","19,384.72",M2044282225,0.00,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,"21,182.00",0.00,1,0
4,1,PAYMENT,"11,668.14",C2048537720,"41,554.00","29,885.86",M1230701703,0.00,0.00,0,0


## Inspecting dataset size and available columns

This step checks two of the most basic structural properties of the dataset:
- the number of rows and columns,
- the list of feature names.

The shape tells us how much data is currently loaded into memory, while the column list tells us what variables are available for future analysis.

This is a critical schema validation step. Before analyzing fraud behavior, we need to confirm that the expected fields are present, including transaction type, balances, account identifiers, and fraud-related labels.

The expected output is:
- a tuple showing the dimensions of the loaded sample,
- a complete list of all column names in the dataset.

If anything is missing or unexpected here, that would indicate a file loading issue, schema mismatch, or incorrect source file.

In [18]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (10000, 11)

Columns:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


## Reviewing data types and completeness at a structural level

This step uses `df.info()` to inspect the inferred data types of each column and the number of non-null values.

This is one of the most important early audit checks because data types affect:
- memory usage,
- correctness of calculations,
- compatibility with preprocessing pipelines,
- feature engineering logic,
- model training later on.

For example:
- numeric columns should be numeric rather than strings,
- categorical fields such as transaction type may later be encoded efficiently,
- binary target columns should be stored in compact integer form,
- identifier columns must be recognized correctly so they are not treated as continuous measurements.

The non-null counts also provide an early signal about missing data. If non-null counts differ across columns, that would suggest incomplete records and may require cleaning or special treatment later.

In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   step            10000 non-null  int32   
 1   type            10000 non-null  category
 2   amount          10000 non-null  float32 
 3   nameOrig        10000 non-null  string  
 4   oldbalanceOrg   10000 non-null  float32 
 5   newbalanceOrig  10000 non-null  float32 
 6   nameDest        10000 non-null  string  
 7   oldbalanceDest  10000 non-null  float32 
 8   newbalanceDest  10000 non-null  float32 
 9   isFraud         10000 non-null  int8    
 10  isFlaggedFraud  10000 non-null  int8    
dtypes: category(1), float32(5), int32(1), int8(2), string(2)
memory usage: 625.0 KB


## Evaluating missing values across all columns

This step calculates the number of missing values in each column and sorts the results in descending order.

The purpose is to identify whether the dataset contains null values and, if so, which fields are affected most heavily.

This matters because missing values can influence:
- descriptive statistics,
- feature construction,
- anomaly detection logic,
- model performance,
- business interpretation of balance-related fields.

Sorting the missing counts from highest to lowest helps prioritize which variables may need attention first. If the dataset shows no missing values, that is still an important finding because it simplifies later preprocessing decisions.

The expected output is a ranked summary of null counts by column.

In [20]:
df.isnull().sum().sort_values(ascending=False)

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

## Producing summary statistics for numeric and categorical fields

This step generates descriptive statistics for all columns using `describe(include="all")`, then transposes the result so each row corresponds to one feature.

The goal is to gain a first-pass statistical profile of the dataset.

For numeric fields, this helps reveal:
- count,
- mean,
- standard deviation,
- minimum and maximum values,
- quartile structure.

For categorical fields, it helps reveal:
- number of unique values,
- most frequent category,
- frequency of the most common category.

This is a high-value audit step because it can expose early anomalies such as:
- suspicious zero-heavy balance fields,
- highly skewed amounts,
- dominant transaction types,
- categorical values that may require closer inspection.

At this point, we are not interpreting every number deeply, but we are establishing an evidence base for later EDA and feature design.

In [21]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
step,"10,000.00",NaN,NaN,NaN,4.18,2.48,1.00,1.00,5.00,7.00,7.00
type,10000,5,PAYMENT,5465,NaN,NaN,NaN,NaN,NaN,NaN,NaN
amount,"10,000.00",NaN,NaN,NaN,"103,546.69","266,307.19",2.39,"4,397.53","12,858.74","114,382.48","10,000,000.00"
nameOrig,10000,10000,C1231006815,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
oldbalanceOrg,"10,000.00",NaN,NaN,NaN,"893,933.06","2,135,683.00",0.00,127.69,"21,375.55","178,271.92","12,930,418.00"
newbalanceOrig,"10,000.00",NaN,NaN,NaN,"915,274.12","2,181,428.50",0.00,0.00,"10,349.94","176,093.44","13,010,503.00"
nameDest,10000,6397,C985934102,62,NaN,NaN,NaN,NaN,NaN,NaN,NaN
oldbalanceDest,"10,000.00",NaN,NaN,NaN,"934,275.81","2,676,340.25",0.00,0.00,0.00,"283,106.69","19,516,116.00"
newbalanceDest,"10,000.00",NaN,NaN,NaN,"1,096,606.38","3,014,496.00",0.00,0.00,0.00,"252,055.23","19,169,204.00"
isFraud,"10,000.00",NaN,NaN,NaN,0.01,0.08,0.00,0.00,0.00,0.00,1.00


## Reviewing value counts for important categorical and fraud-related variables

This step calculates value counts for three strategically important columns:
- `type`, which represents transaction category,
- `isFraud`, which is the primary fraud label,
- `isFlaggedFraud`, which appears to be a system-generated fraud flag.

This step matters because these variables define the high-level operating landscape of the dataset.

For `type`, we want to understand the composition of transaction categories and which business actions dominate the data.

For `isFraud`, we want to assess class imbalance, which is one of the most important modeling constraints in fraud detection. If fraud cases are extremely rare, then later model evaluation must go beyond simple accuracy.

For `isFlaggedFraud`, we want to understand how often an existing rule-based system appears to flag fraud. This variable may be useful for benchmarking, but it must be handled carefully to avoid leakage or artificial model inflation.

The expected output is a count distribution for each of these columns.

In [22]:
for col in ["type", "isFraud", "isFlaggedFraud"]:
    print(f"\nValue counts for {col}:")
    print(df[col].value_counts(dropna=False))


Value counts for type:
type
PAYMENT     5465
CASH_IN     1949
CASH_OUT    1321
TRANSFER     921
DEBIT        344
Name: count, dtype: int64

Value counts for isFraud:
isFraud
0    9932
1      68
Name: count, dtype: int64

Value counts for isFlaggedFraud:
isFlaggedFraud
0    10000
Name: count, dtype: int64


## Auditing the time variable and estimating temporal coverage

This step checks the minimum and maximum values of the `step` column and then uses that range to estimate how many hours and days are covered by the sample.

The `step` variable is strategically important because this project is not only about fraud classification, but also about time-based fraud monitoring and behavior analysis.

Understanding the time field early helps us answer questions such as:
- What is the temporal span of the dataset?
- Is time represented in hourly increments?
- Can we later aggregate fraud activity by hour, day, or rolling window?
- Is the dataset suitable for time-aware feature engineering and trend analysis?

The expected output includes:
- the minimum observed step,
- the maximum observed step,
- the approximate number of hours covered,
- the approximate number of days represented.

This step lays the groundwork for later temporal EDA and time-based risk monitoring.

In [23]:
print("Min step:", df["step"].min())
print("Max step:", df["step"].max())
print("Approx hours covered:", df["step"].max() - df["step"].min() + 1)
print("Approx days covered:", (df["step"].max() - df["step"].min() + 1) / 24)

Min step: 1
Max step: 7
Approx hours covered: 7
Approx days covered: 0.2916666666666667


## Initial audit summary

The dataset sample loaded successfully and contains 10,000 rows and 11 columns, which matches the expected transaction-level structure of the fraud dataset.

Several important findings emerged from this first audit:

- All 11 expected columns are present, including transaction metadata, account identifiers, balance fields, the fraud label (`isFraud`), and the system flag field (`isFlaggedFraud`).
- The data types appear well-optimized for analysis: numeric fields use compact integer and float formats, transaction type is stored as a categorical variable, and account identifiers are stored as strings.
- No missing values were found in any column within this 10,000-row sample. This suggests that null handling may not be a major issue at the initial stage, although this should still be confirmed on the full dataset.
- The transaction type distribution is highly uneven. `PAYMENT` is the dominant class, followed by `CASH_IN`, while `DEBIT` is relatively rare.
- The fraud label is strongly imbalanced: 68 out of 10,000 transactions are labeled as fraud, while 9,932 are non-fraud. This confirms that class imbalance will be a major modeling challenge.
- The `isFlaggedFraud` field contains only zeros in this sample, indicating that the system-generated fraud flag is either extremely rare or not active in the early portion of the dataset.
- The sampled records cover steps 1 through 7 only, which corresponds to approximately 7 hours, or about 0.29 days. This means the current sample only captures a very small early time window and should not yet be treated as representative of the dataset’s full temporal behavior.

Overall, the sample confirms that the dataset is structurally clean and suitable for deeper analysis. However, the heavy class imbalance, uneven transaction-type distribution, and narrow time coverage indicate that the next phase should move beyond basic schema inspection and focus on validating business logic, exploring suspicious transaction patterns, and expanding the temporal perspective.